# 01 — Baseline Inference (BLIP VQA)

**Goal:** verify that the pretrained `Salesforce/blip-vqa-base` model runs correctly
and produces an answer for an (image, question) pair.

**Run this on Google Colab with a T4 GPU** (Runtime → Change runtime type → T4 GPU).

This notebook does **not** train anything — it only loads pretrained weights.

In [ ]:
# If running on a fresh Colab runtime, install dependencies.
# On your own machine with requirements.txt already installed, skip this cell.
!pip install -q transformers torch torchvision pillow requests

In [ ]:
import sys, os
# When this notebook lives in VisionQuery/notebooks/, add the project root
# to sys.path so `from src...` imports work the same as running scripts.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
# Record these values into configs/config.yaml's reproducibility section
# once you know them, replacing the [TO BE RECORDED IN COLAB] placeholders.

**Expected output:** version strings, and `CUDA available: True` with a `Tesla T4` line if the Colab GPU runtime is active. If `False`, the code will still run on CPU (slower).

In [ ]:
from src.model import load_model, get_device

processor, model, device, used_finetuned = load_model(model_name="Salesforce/blip-vqa-base")
print("Loaded on device:", device)
print("Using fine-tuned checkpoint:", used_finetuned)  # should be False here - no checkpoint exists yet

**Common errors & fixes:**\n- `OSError: Can't load ... from Hub` → check your internet connection / HF Hub isn't rate-limiting; retry.\n- Model download is slow the first time (~1-2GB) — this is normal, it's cached afterwards.

In [ ]:
import requests
from PIL import Image
from io import BytesIO
from src.inference import answer_question

# A small public test image (replace with your own upload if you like)
url = "https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg"
image = Image.open(BytesIO(requests.get(url).content)).convert("RGB")
image

In [ ]:
question = "how many dogs are in the picture?"
answer = answer_question(image, question, processor, model, device)
print("Q:", question)
print("A:", answer)

**Expected output:** a short answer string (e.g. a number or a word). This confirms baseline inference works end-to-end. **This is a real, executed result once you run this cell in Colab — do not copy an example answer without running it yourself.**\n\n**Next notebook:** `02_dataset_preparation.ipynb`